# YouTube → Mel-RoFormer → Zero-Contamination Experiment

This notebook mirrors the SonicStudio **Experiment** funnel, with one processing step per cell. Run it on the model server (`vsf-242`) with the repository `.venv` kernel. Downloads, stems, previews, and the saved diarization result stay under the repository `.data/` directory.

> The default configuration deliberately enables every optional Experiment gate. Stage 5a needs either `GEMINI_API_KEY` or a reachable local Gemma endpoint. Stage 5b loads VibeVoice locally unless `VIBEVOICE_ENDPOINT` is set.

In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from html import escape as html_escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio as IPythonAudio, HTML, clear_output, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

# Works when Jupyter starts in src/notebooks/ as documented, and also from repo subdirectories.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Could not find the repository root (pyproject.toml).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.data_paths import DATA_DIR
from src.utils.AudioClass import Audio
from src.utils.AudioCutter import AudioCutter
from src.yt_crawler.YtCrawlerClass import YtCrawler
from src.separation import MelRoFormer
from src.diarization import (
    DiariZenWorkerDiarizer,
    DiarizationModelInfo,
    DiarizationResult,
    SortformerWorkerDiarizer,
    Speaker,
    ZeroContaminationConfig,
)
from src.diarization.zero_contamination import (
    align_and_lock_syllable_boundaries,
    apply_context_aware_collar,
    compute_consensus_turns,
    filter_by_embedding_homogeneity,
    filter_by_foundation_models,
    snap_boundaries_to_acoustic_valleys,
)


## Configuration

Set the URL and devices before running. The values below follow the Experiment tab defaults while enabling all optional gates.

In [ ]:
URL = "https://www.youtube.com/watch?v=REPLACE_ME"
DEVICE = "cuda:0"
SECONDARY_DEVICE = DEVICE
HOMOGENEITY_DEVICE = DEVICE
ALIGNER_DEVICE = "cpu"
VIBEVOICE_DEVICE = "cuda:1"
DIRECT_AUDIO_BACKEND = os.getenv("DIRECT_AUDIO_BACKEND", "gemini")
DIRECT_AUDIO_MODEL = os.getenv("DIRECT_AUDIO_MODEL") or (
    "gemini-3.8-flash" if DIRECT_AUDIO_BACKEND == "gemini" else None
)

config = ZeroContaminationConfig(
    device=DEVICE,
    primary_device=DEVICE,
    primary_backend="sortformer",
    target_onset=0.80,
    target_offset=0.65,
    enable_consensus=True,
    secondary_backend="diarizen",
    secondary_device=SECONDARY_DEVICE,
    enable_collar_erosion=True,
    boundary_collar_s=0.35,
    min_turn_duration_s=0.80,
    transition_exclusion_s=0.50,
    enable_context_collar=True,
    handoff_risk_distance_s=0.80,
    silence_tail_buffer_s=0.027,
    enable_syllable_alignment=True,
    aligner_engine="whisper_timestamped",
    aligner_model="vinai/PhoWhisper-small",
    aligner_language="vi",
    aligner_device=ALIGNER_DEVICE,
    enable_energy_snapping=True,
    energy_search_window_s=0.15,
    energy_valley_floor_db=-30.0,
    energy_frame_len_ms=2.0,
    energy_hop_len_ms=0.5,
    enable_homogeneity=True,
    homogeneity_device=HOMOGENEITY_DEVICE,
    homogeneity_window_s=1.0,
    homogeneity_hop_s=0.25,
    min_homogeneity_similarity=0.75,
    enable_gemma=True,
    gemma_backend=DIRECT_AUDIO_BACKEND,
    gemma_endpoint=os.getenv("UNSLOTH_ENDPOINT"),
    gemma_model=DIRECT_AUDIO_MODEL,
    gemma_api_key=os.getenv("GEMINI_API_KEY"),
    enable_vibevoice=True,
    vibevoice_device=VIBEVOICE_DEVICE,
    vibevoice_endpoint=os.getenv("VIBEVOICE_ENDPOINT"),
    max_secondary_speech_s=0.0,
    token=os.getenv("HF_TOKEN"),
)

stage_stats = {}
def record_stage(name, turns):
    turns = list(turns)
    stage_stats[name] = {
        "turns": len(turns),
        "speech_duration_s": round(sum(turn.duration_s for turn in turns), 2),
    }
    display(stage_stats[name])
    return turns


## Input — crawl the URL

In [ ]:
crawler = YtCrawler(
    output_dir=DATA_DIR / "notebook" / "zero_contamination" / "downloads",
    work_dir=DATA_DIR / "notebook" / "zero_contamination" / "crawl_work",
)
source_audio: Audio = crawler.download(URL)
display(source_audio.metadata())
source_audio.notebook_display()


## Separation — Mel-RoFormer vocals stem

In [ ]:
separator = MelRoFormer(
    device=DEVICE,
    two_stems="vocals",
    output_dir=DATA_DIR / "notebook" / "zero_contamination" / "stems",
    work_dir=DATA_DIR / "notebook" / "zero_contamination" / "separation_work",
)
with separator:
    speech_audio: Audio = separator.separate(source_audio)
display(speech_audio.metadata())
speech_audio.notebook_display()


## Experiment Stage 1 — primary Sortformer diarization

In [ ]:
primary_diarizer = SortformerWorkerDiarizer(
    device=config.primary_device or config.device,
    token=config.token,
    onset=config.target_onset,
    offset=config.target_offset,
)
with primary_diarizer:
    primary_result: DiarizationResult = primary_diarizer.diarize(speech_audio)
current_turns = record_stage("1_primary", sorted(primary_result.turns, key=lambda turn: turn.start_s))


## Experiment Stage 2 — DiariZen + Hungarian mutual consensus

In [ ]:
secondary_diarizer = DiariZenWorkerDiarizer(
    device=config.secondary_device or config.device,
    token=config.token,
)
with secondary_diarizer:
    secondary_result: DiarizationResult = secondary_diarizer.diarize(speech_audio)
current_turns, speaker_mapping = compute_consensus_turns(
    current_turns, secondary_result.turns, speech_audio.duration_s
)
current_turns = record_stage("2_consensus", current_turns)
display({"speaker_mapping": speaker_mapping})


## Experiment Stage 3a — context-aware collar and handoff guard

In [ ]:
current_turns, collar_audits = apply_context_aware_collar(
    current_turns,
    collar_s=config.boundary_collar_s,
    handoff_risk_s=config.handoff_risk_distance_s,
    silence_tail_s=config.silence_tail_buffer_s,
    min_duration_s=config.min_turn_duration_s,
    transition_exclusion_s=config.transition_exclusion_s,
    audio_duration_s=speech_audio.duration_s,
)
current_turns = record_stage("3a_context_collar", current_turns)


## Experiment Stage 3b — syllable/word forced-alignment lock

In [ ]:
current_turns, alignment_audits = align_and_lock_syllable_boundaries(
    speech_audio,
    current_turns,
    aligner_engine=config.aligner_engine,
    aligner_model=config.aligner_model,
    aligner_language=config.aligner_language,
    aligner_endpoint=config.aligner_endpoint,
    aligner_device=config.aligner_device or "cpu",
    token=config.token,
)
current_turns = record_stage("3b_word_lock", current_turns)


## Experiment Stage 3c — micro-energy valley snapping

In [ ]:
current_turns, energy_audits = snap_boundaries_to_acoustic_valleys(
    speech_audio,
    current_turns,
    search_window_s=config.energy_search_window_s,
    energy_floor_db=config.energy_valley_floor_db,
    frame_len_ms=config.energy_frame_len_ms,
    hop_len_ms=config.energy_hop_len_ms,
)
current_turns = record_stage("3c_energy_snap", current_turns)


## Experiment Stage 4 — WeSpeaker sliding-window homogeneity

In [ ]:
current_turns, homogeneity_audits = filter_by_embedding_homogeneity(
    speech_audio,
    current_turns,
    window_s=config.homogeneity_window_s,
    hop_s=config.homogeneity_hop_s,
    min_similarity=config.min_homogeneity_similarity,
    device=config.homogeneity_device or config.device,
    token=config.token,
)
current_turns = record_stage("4_homogeneity", current_turns)


## Experiment Stage 5a — Gemma/Gemini direct-audio verifier

This fails closed when the configured verifier is unavailable, matching the Experiment pipeline.

In [ ]:
direct_audio_config = replace(config, enable_gemma=True, enable_vibevoice=False)
current_turns, direct_audio_audits = filter_by_foundation_models(
    speech_audio, current_turns, direct_audio_config
)
current_turns = record_stage("5a_direct_audio", current_turns)


## Experiment Stage 5b — VibeVoice-ASR speaker-count verifier

In [ ]:
vibevoice_config = replace(config, enable_gemma=False, enable_vibevoice=True)
current_turns, vibevoice_audits = filter_by_foundation_models(
    speech_audio, current_turns, vibevoice_config
)
current_turns = record_stage("5b_vibevoice", current_turns)


## Assemble and persist the canonical diarization result

In [ ]:
speaker_ids = sorted({turn.speaker_id for turn in current_turns})
diarization_result = DiarizationResult(
    schema_version="2.0",
    audio_id=speech_audio.source_id,
    speakers=[Speaker(speaker_id=speaker_id) for speaker_id in speaker_ids],
    turns=current_turns,
    source_audio=speech_audio,
    model=DiarizationModelInfo(
        backend="zero-contamination-notebook",
        model_id="sortformer+diarizen+all-experiment-gates",
    ),
)
result_path = diarization_result.save(
    DATA_DIR / "notebook" / "zero_contamination" / "results"
)
display({"saved_to": str(result_path), "funnel": stage_stats})


## Diarization result notebook viewer

The viewer brings the Experiment result table's speaker filtering, transcript search, raw/refined boundary comparison, waveform context, and lazy per-turn audio playback into Jupyter.

In [ ]:
class DiarizationResultNotebookViewer:
    """Interactive Jupyter viewer for a file-backed ``DiarizationResult``.

    Turn clips are created lazily under ``.data/notebook/diarization_viewer``.
    The controls mirror SonicStudio's result viewer: filter by speaker or
    text, inspect boundary metadata, compare blunt/refined audio, and view
    the waveform around the selected turn.
    """

    def __init__(
        self,
        result: DiarizationResult,
        output_dir: str | Path = DATA_DIR / "notebook" / "diarization_viewer",
    ) -> None:
        if result.source_audio is None:
            raise ValueError("The diarization result has no source_audio.")
        if not Path(result.source_audio.path).is_file():
            raise FileNotFoundError(result.source_audio.path)
        self.result = result
        self.audio = result.source_audio
        self.output_dir = Path(output_dir) / result.result_id
        self.cutter = AudioCutter(output_dir=self.output_dir)
        self._filtered_indices: list[int] = []

        speakers = ["All speakers", *sorted({t.speaker_id for t in result.turns})]
        self.speaker = widgets.Dropdown(options=speakers, description="Speaker:")
        self.search = widgets.Text(description="Search:", placeholder="transcript or policy")
        self.turn = widgets.Dropdown(options=[], description="Turn:", layout=widgets.Layout(width="100%"))
        self.summary = widgets.HTML()
        self.detail = widgets.Output()

        self.speaker.observe(self._filters_changed, names="value")
        self.search.observe(self._filters_changed, names="value")
        self.turn.observe(self._turn_changed, names="value")
        self._refresh_filters()

    @staticmethod
    def _transcript(turn) -> str:
        return str(getattr(turn, "_transcript", getattr(turn, "transcript", "")) or "")

    @staticmethod
    def _policy(turn) -> str:
        return str(getattr(turn, "_boundary_policy", "standard"))

    def _filters_changed(self, _change=None) -> None:
        self._refresh_filters()

    def _refresh_filters(self) -> None:
        wanted_speaker = self.speaker.value
        query = self.search.value.strip().lower()
        matches = []
        for index, turn in enumerate(self.result.turns):
            if wanted_speaker != "All speakers" and turn.speaker_id != wanted_speaker:
                continue
            haystack = f"{turn.speaker_id} {self._transcript(turn)} {self._policy(turn)}".lower()
            if query and query not in haystack:
                continue
            matches.append(index)
        self._filtered_indices = matches
        duration = sum(self.result.turns[index].duration_s for index in matches)
        average = duration / len(matches) if matches else 0.0
        self.summary.value = (
            f"<b>{len(matches)}</b> clean turns &nbsp;•&nbsp; "
            f"<b>{duration:.1f}s</b> speech &nbsp;•&nbsp; average <b>{average:.1f}s</b>"
        )
        options = [
            (f"#{index + 1} · {self.result.turns[index].speaker_id} · "
             f"{self.result.turns[index].start_s:.2f}–{self.result.turns[index].end_s:.2f}s", index)
            for index in matches
        ]
        self.turn.options = options
        if not options:
            with self.detail:
                clear_output(wait=True)
                display(HTML("<i>No turns match the active filters.</i>"))

    def _turn_changed(self, change) -> None:
        if change.get("new") is not None:
            self._render_turn(int(change["new"]))

    def _clip(self, index: int, start_s: float, end_s: float, kind: str) -> Audio:
        path = self.output_dir / f"turn_{index:06d}_{kind}.wav"
        if path.is_file():
            return Audio.from_file(path)
        return self.cutter.cut(self.audio, start_s, end_s, output_path=path)

    def _waveform(self, start_s: float, end_s: float, raw_start_s: float, raw_end_s: float) -> None:
        context_start = max(0.0, min(start_s, raw_start_s) - 0.20)
        context_end = min(self.audio.duration_s, max(end_s, raw_end_s) + 0.20)
        with sf.SoundFile(str(self.audio.path)) as source:
            sample_rate = int(source.samplerate)
            source.seek(int(context_start * sample_rate))
            waveform = source.read(int((context_end - context_start) * sample_rate), dtype="float32", always_2d=True)
        mono = waveform.mean(axis=1) if len(waveform) else np.zeros(1, dtype=np.float32)
        stride = max(1, len(mono) // 5000)
        times = context_start + np.arange(0, len(mono), stride) / sample_rate
        figure, axis = plt.subplots(figsize=(12, 2.4))
        axis.plot(times, mono[::stride], color="#64748b", linewidth=0.7)
        axis.axvspan(raw_start_s, raw_end_s, color="#f59e0b", alpha=0.18, label="Blunt/raw")
        axis.axvspan(start_s, end_s, color="#10b981", alpha=0.20, label="Refined")
        axis.set(xlabel="Time (s)", ylabel="Amplitude", xlim=(context_start, context_end))
        axis.legend(loc="upper right")
        figure.tight_layout()
        plt.show()
        plt.close(figure)

    def _render_turn(self, index: int) -> None:
        turn = self.result.turns[index]
        raw_start = float(getattr(turn, "_raw_start_s", turn.start_s))
        raw_end = float(getattr(turn, "_raw_end_s", turn.end_s))
        delta_end = float(getattr(turn, "_delta_end_ms", 0.0))
        transcript = self._transcript(turn) or "—"
        metadata = (
            "<table style='width:100%;text-align:left'>"
            f"<tr><th>Speaker</th><td>{html_escape(turn.speaker_id)}</td><th>Duration</th><td>{turn.duration_s:.2f}s</td></tr>"
            f"<tr><th>Refined</th><td>{turn.start_s:.3f}–{turn.end_s:.3f}s</td><th>Raw/blunt</th><td>{raw_start:.3f}–{raw_end:.3f}s</td></tr>"
            f"<tr><th>Policy</th><td>{html_escape(self._policy(turn))}</td><th>End delta</th><td>{delta_end:+.0f}ms</td></tr>"
            f"<tr><th>Transcript</th><td colspan='3'>{html_escape(transcript)}</td></tr></table>"
        )
        refined = self._clip(index, turn.start_s, turn.end_s, "refined")
        raw = self._clip(index, raw_start, raw_end, "raw")
        with self.detail:
            clear_output(wait=True)
            display(HTML(metadata))
            self._waveform(turn.start_s, turn.end_s, raw_start, raw_end)
            display(HTML("<b>Refined boundary</b>"))
            display(IPythonAudio(filename=str(refined.path)))
            if raw_start != turn.start_s or raw_end != turn.end_s:
                display(HTML("<b>Raw/blunt boundary</b>"))
                display(IPythonAudio(filename=str(raw.path)))

    def display(self) -> None:
        """Render the interactive viewer once and return ``None``."""
        controls = widgets.HBox([self.speaker, self.search])
        display(widgets.VBox([controls, self.summary, self.turn, self.detail]))
        if self.turn.value is not None:
            self._render_turn(int(self.turn.value))


viewer = DiarizationResultNotebookViewer(diarization_result)
viewer.display()
